In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types as t
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder\
        .config("spark.master", "local[3]")\
        .config("spark.app.name", "AprendendoDF")\
        .config("spark.executor.memory", "450m")\
        .config("spark.cores.max","3")\
        .config("spark.sql.shuffle.partitions", "20")\
        .getOrCreate()
spark

Definindo a estrutura do esquema

In [3]:
schema = t.StructType([
    t.StructField("longitude", t.DoubleType(), nullable=True),
    t.StructField("latitude", t.DoubleType(), nullable=True),
    t.StructField("housing_median_age", t.DoubleType(), nullable=True),
    t.StructField("total_rooms", t.DoubleType(), nullable=True),
    t.StructField("total_bedrooms", t.DoubleType(), nullable=True),
    t.StructField("population", t.DoubleType(), nullable=True),
    t.StructField("households", t.DoubleType(), nullable=True),
    t.StructField("median_income", t.DoubleType(), nullable=True),
    t.StructField("median_house_value", t.DoubleType(), nullable=True),
    t.StructField("ocean_proximity", t.StringType(), nullable=True),
])
schema

StructType([StructField('longitude', DoubleType(), True), StructField('latitude', DoubleType(), True), StructField('housing_median_age', DoubleType(), True), StructField('total_rooms', DoubleType(), True), StructField('total_bedrooms', DoubleType(), True), StructField('population', DoubleType(), True), StructField('households', DoubleType(), True), StructField('median_income', DoubleType(), True), StructField('median_house_value', DoubleType(), True), StructField('ocean_proximity', StringType(), True)])

* [Type de Pyspark](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)
* [Leitura](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html) 

In [4]:
data = spark.read\
    .format("csv")\
    .csv(
        path="C:\\Users\\mateu\\Documents\\Norton\\Projetos GIT\\learn-databricks\\dataset\\housing.csv",
        schema=schema,
        sep=',',
        header=True
    )

In [5]:
data.show(3)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
only showing top 3 rows


In [6]:
from pyspark.sql import functions as F

> [Functions cols pyspark](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.col.html)

Renomenado coluna

In [7]:
data\
    .select(
        F.col("longitude").alias("Lon")
    ).show(2)

+-------+
|    Lon|
+-------+
|-122.23|
|-122.22|
+-------+
only showing top 2 rows


In [8]:
data.show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

Between coluna

In [9]:
data\
    .select(F.col(col="housing_median_age").between(lowerBound=20, upperBound=40))\
    .show(3)

+-----------------------------------------------------------+
|((housing_median_age >= 20) AND (housing_median_age <= 40))|
+-----------------------------------------------------------+
|                                                      false|
|                                                       true|
|                                                      false|
+-----------------------------------------------------------+
only showing top 3 rows


Dividindo a coluna

In [10]:
data.\
    select(F.try_divide(F.col("housing_median_age"), F.col("population"))).\
    where(F.try_divide(F.col("housing_median_age"), F.col("population"))>=1).\
    show(3)

+------------------------------------------+
|try_divide(housing_median_age, population)|
+------------------------------------------+
|                        2.5555555555555554|
|                        1.2432432432432432|
|                         1.434782608695652|
+------------------------------------------+
only showing top 3 rows


In [11]:
data.select(
    F.when(F.col('population')<=100, 1)\
    .otherwise(2)
).show()

+-----------------------------------------------+
|CASE WHEN (population <= 100) THEN 1 ELSE 2 END|
+-----------------------------------------------+
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|
|                                              2|


### Agrupamento

In [12]:
data.groupBy("ocean_proximity").count().show()

+---------------+-----+
|ocean_proximity|count|
+---------------+-----+
|      <1H OCEAN| 9136|
|         INLAND| 6551|
|         ISLAND|    5|
|     NEAR OCEAN| 2658|
|       NEAR BAY| 2290|
+---------------+-----+



In [13]:
data.summary().show()

+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|summary|          longitude|         latitude|housing_median_age|       total_rooms|    total_bedrooms|        population|       households|     median_income|median_house_value|ocean_proximity|
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|  count|              20640|            20640|             20640|             20640|             20433|             20640|            20640|             20640|             20640|          20640|
|   mean|-119.56970445736148| 35.6318614341087|28.639486434108527|2635.7630813953488| 537.8705525375618|1425.4767441860465|499.5396802325581|3.8706710029070246|206855.81690891474|           NULL|
| stddev|  2.0035317

In [21]:
data.createOrReplaceTempView("data")

In [23]:
spark.sql("SELECT * FROM data").show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              